# XNav Colab ISO Builder v1.1.0 (C++ edition)

This notebook builds a flashable **XNav** `.img.xz` image entirely inside Google Colab — no local Linux machine, WSL, or root access required.

The build uses `guestfish` (from `libguestfs-tools`) to manipulate disk images in userspace, making it compatible with Colab's containerised runtime.

**What you get:**
- A ready-to-flash Raspberry Pi OS image with XNav pre-installed
- Hostname set to `xnav` (accessible at `http://xnav.local:5800`)
- On first boot the C++ vision binary is compiled automatically (~5–10 min, requires internet)

**Prerequisites:**
- Google account (free tier works; Colab Pro recommended for longer runtimes)
- ≥ 10 GB free Colab disk space (default provides ~80 GB)
- Internet connection
- ~30–60 minutes depending on network speed and compression

---
> **Tip:** Run cells top-to-bottom in order. If the runtime disconnects during compression (Step 5), re-mount Google Drive and the partially-built image may still be there.

## Step 1 — Clone the XNav Repository

In [ ]:
import os

REPO_DIR = "/content/Limelight3-XNav"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/realaaravdas/Limelight3-XNav.git {REPO_DIR}
else:
    print(f"Repository already cloned at {REPO_DIR}")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print("\nRepository ready.")

## Step 2 — Download the Base Raspberry Pi OS Image

This downloads Raspberry Pi OS Lite (64-bit, ~500 MB). Skip this cell and re-run if you already have the image cached from a previous session.

In [ ]:
import os, subprocess

WORK_DIR = "/tmp/xnav-build"
os.makedirs(WORK_DIR, exist_ok=True)

base_img = os.path.join(WORK_DIR, "raspios_lite_arm64_latest.img.xz")

if not os.path.exists(base_img):
    print("Downloading Raspberry Pi OS Lite (64-bit ARM) — this takes ~5 minutes...")
    result = subprocess.run([
        "wget", "-q", "--show-progress",
        "-O", base_img,
        "https://downloads.raspberrypi.org/raspios_lite_arm64_latest"
    ])
    if result.returncode == 0:
        size_mb = os.path.getsize(base_img) // (1024 * 1024)
        print(f"Download complete. Size: {size_mb} MB")
    else:
        print("ERROR: Download failed. Check your internet connection and retry.")
else:
    size_mb = os.path.getsize(base_img) // (1024 * 1024)
    print(f"Base image already present ({size_mb} MB). Skipping download.")

## Step 3 — (Optional) Mount Google Drive

Mount your Google Drive **before** the build so the finished image is saved automatically. This avoids losing the image if Colab disconnects during or after compression.

Skip this cell if you prefer to download the image directly from Colab after the build.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    GDRIVE_SAVE_PATH = "/content/drive/MyDrive/xnav-1.1.0.img.xz"
    print(f"Google Drive mounted. Image will be saved to: {GDRIVE_SAVE_PATH}")
except Exception as e:
    GDRIVE_SAVE_PATH = None
    print(f"Google Drive not available or skipped: {e}")
    print("Image will only be available at /tmp/xnav-build/xnav-1.1.0.img.xz")

## Step 4 — Run the XNav Colab Build Script

This is the main build step. It will:
1. Install `libguestfs-tools`, `qemu-utils`, and other dependencies
2. Decompress and copy the base Raspberry Pi OS image
3. Expand the image by 512 MB and resize the root filesystem
4. Stage all XNav files (web dashboard, C++ source, config, services)
5. Download Bootstrap CSS/JS for the offline web dashboard
6. Inject everything into the image via `guestfish`
7. Set hostname to `xnav`, configure boot parameters, enable systemd services
8. Compress the final image with `xz -9`

**Expected duration:** ~30–60 minutes

> The compression step alone (xz -9) can take 10–20 minutes. Do not close the tab.

In [ ]:
import subprocess, sys, os

REPO_DIR = "/content/Limelight3-XNav"
BUILD_SCRIPT = os.path.join(REPO_DIR, "system/scripts/colab_build.sh")

if not os.path.exists(BUILD_SCRIPT):
    print(f"ERROR: Build script not found at {BUILD_SCRIPT}")
    print("Make sure Step 1 (clone) completed successfully.")
else:
    os.chmod(BUILD_SCRIPT, 0o755)
    print(f"Starting build: {BUILD_SCRIPT}")
    print("=" * 60)
    result = subprocess.run(
        ["bash", BUILD_SCRIPT],
        cwd=REPO_DIR
    )
    print("=" * 60)
    if result.returncode == 0:
        print("\nBuild completed successfully!")
    else:
        print(f"\nERROR: Build script exited with code {result.returncode}")
        print("Check the output above for error details.")
        sys.exit(result.returncode)

## Step 5 — Verify and Save the Output Image

In [ ]:
import os, shutil

OUTPUT_IMG = "/tmp/xnav-build/xnav-1.1.0.img.xz"

if os.path.exists(OUTPUT_IMG):
    size_mb = os.path.getsize(OUTPUT_IMG) // (1024 * 1024)
    print(f"Build output: {OUTPUT_IMG}")
    print(f"Image size: {size_mb} MB")
else:
    print("ERROR: Output image not found. Check build output above for errors.")

# Save to Google Drive if mounted
try:
    if globals().get('GDRIVE_SAVE_PATH') and os.path.exists("/content/drive"):
        print(f"\nCopying to Google Drive: {GDRIVE_SAVE_PATH} ...")
        shutil.copy2(OUTPUT_IMG, GDRIVE_SAVE_PATH)
        saved_mb = os.path.getsize(GDRIVE_SAVE_PATH) // (1024 * 1024)
        print(f"Saved to Google Drive ({saved_mb} MB). Safe to close Colab now.")
    else:
        print("\nGoogle Drive not mounted. Download the image directly from Colab (next cell).")
except Exception as e:
    print(f"\nWARN: Could not save to Google Drive: {e}")
    print("Download the image directly using the next cell.")

## Step 6 — Download the Image to Your Computer

Run this cell to download `xnav-1.1.0.img.xz` directly to your browser's Downloads folder.

> **Note:** This only works while the Colab runtime is still active. If you saved to Google Drive in Step 5 you can skip this cell and download from Drive instead.

In [ ]:
import os

OUTPUT_IMG = "/tmp/xnav-build/xnav-1.1.0.img.xz"

if os.path.exists(OUTPUT_IMG):
    try:
        from google.colab import files
        print("Initiating download...")
        files.download(OUTPUT_IMG)
    except ImportError:
        print("Not running in Google Colab. Copy the file manually from:")
        print(f"  {OUTPUT_IMG}")
else:
    print("ERROR: Image not found. Run Step 4 first.")

## Step 7 — Flash the Image

Download `xnav-1.1.0.img.xz` from Google Drive (or directly from Step 6) to your local machine, then flash it using one of these methods:

### Option A: balenaEtcher (recommended — cross-platform)
1. Download [balenaEtcher](https://etcher.balena.io/)
2. **Flash from file** → select `xnav-1.1.0.img.xz`
3. **Select target** → choose the Limelight 3 eMMC / SD card
4. **Flash!**

### Option B: Raspberry Pi Imager
1. Open [Raspberry Pi Imager](https://www.raspberrypi.com/software/)
2. **Operating System** → **Use custom** → select `xnav-1.1.0.img.xz`
3. **Storage** → select the target drive
4. Click **Write**

### Option C: Command line (Linux/macOS)
```bash
xzcat xnav-1.1.0.img.xz | sudo dd of=/dev/sdX bs=4M status=progress conv=fsync
```
Replace `/dev/sdX` with the correct target device.

---

## Step 8 — First Boot

1. Connect the Limelight 3 to your robot network via Ethernet
2. Apply power
3. On **first boot**, the device automatically compiles the C++ binary (~5–10 min, requires internet)
4. After compilation finishes, the XNav vision service starts automatically
5. Open a browser and navigate to:
   - `http://xnav.local:5800` (mDNS)
   - or `http://10.TE.AM.11:5800` (replace `TE.AM` with your team number)

> **Note:** Because Google Colab cannot cross-compile ARM64 binaries via QEMU chroot, the C++ binary is compiled on the device on first boot. Subsequent boots start instantly.

---

## Troubleshooting

| Problem | Solution |
|---------|----------|
| `guestfish` hangs or crashes | Ensure `linux-image-generic` is installed (the build script does this). Restart the Colab runtime and re-run from Step 4. |
| Download fails in Step 2 | Download Raspberry Pi OS Lite (64-bit) manually from [raspberrypi.com](https://www.raspberrypi.com/software/operating-systems/) and save it to `/tmp/xnav-build/raspios_lite_arm64_latest.img.xz` |
| Colab disconnects during build | Use **Colab Pro** for longer runtimes. Pre-download the base image (Step 2) before re-running. |
| `libguestfs` supermin errors | The script sets `LIBGUESTFS_BACKEND=direct` and `force_tcg` automatically. Try restarting the runtime. |
| Image too large for Google Drive | Compressed image is ~350–500 MB. Ensure you have enough Drive quota. |
| First-boot compile fails on device | SSH into the Pi and run `sudo bash /opt/xnav/build_on_device.sh` manually. |